In [29]:
import numpy as np
import pandas as pd
import astropy.units as u
from astropy.coordinates import SkyCoord, Longitude
from sunpy.coordinates import get_earth, HeliographicCarrington
from sunpy.coordinates.ephemeris import get_horizons_coord
from solarmach import backmapping_angle, get_sw_speed

#### 1. Get SC Ephemeris
Return the `.obstime`, `(lon, lat, radius)`
- lon & lat: lon & lat separation to Earth lon & lat


In [13]:
# Bulk fetch Solar Orbiter positions (single HORIZONS request)
so_coords = get_horizons_coord('Solar Orbiter', time={
    'start': '2022-03-02 06:00',
    'stop': '2022-03-07 00:00',
    'step': '12h'
})
print(so_coords)

INFO: Obtained JPL HORIZONS location for Solar Orbiter (spacecraft) (-144) [sunpy.coordinates.ephemeris]
<SkyCoord (HeliographicStonyhurst: obstime=['2022-03-02T06:00:00.000' '2022-03-02T18:00:00.000'
 '2022-03-03T06:00:00.000' '2022-03-03T18:00:00.000'
 '2022-03-04T06:00:00.000' '2022-03-04T18:00:00.000'
 '2022-03-05T06:00:00.000' '2022-03-05T18:00:00.000'
 '2022-03-06T06:00:00.000' '2022-03-06T18:00:00.000'], rsun=695700.0 km): (lon, lat, radius) in (deg, deg, AU)
    [(-7.48536023, -4.08465303, 0.56109959),
     (-6.8812709 , -4.11185091, 0.55491756),
     (-6.25192114, -4.13807262, 0.54870727),
     (-5.59633026, -4.16322126, 0.54247053),
     (-4.91347445, -4.18719185, 0.53620932),
     (-4.20228523, -4.2098707 , 0.52992576),
     (-3.46164807, -4.23113471, 0.52362219),
     (-2.69040098, -4.25085064, 0.51730111),
     (-1.88733331, -4.26887438, 0.51096526),
     (-1.05118474, -4.28505014, 0.50461759)]>


so_carr contains lon & lat in a Carrington coordiante system centered on the Sun.
- lon & lat: the carrington degree of the Sun connecting SC and the Sun.

In [27]:
# Transform to Carrington (observer='Sun', matching SolarMACH convention)
so_carr = so_coords.transform_to(HeliographicCarrington(observer='Sun'))

print(f"Fetched {len(so_carr.obstime)} timestamps: {so_carr.obstime[0].iso} to {so_carr.obstime[-1].iso}")
print(f"Lon range: {so_carr.lon.min():.1f} to {so_carr.lon.max():.1f}")
print(f"Dist range: {so_carr.radius.to(u.AU).min():.3f} to {so_carr.radius.to(u.AU).max():.3f}")

Fetched 10 timestamps: 2022-03-02 06:00:00.000 to 2022-03-06 18:00:00.000
Lon range: 8.7 deg to 61.6 deg
Dist range: 0.505 AU to 0.561 AU


Get the real Vsw
- Use speasy package to get Vsw of different SCs

In [ ]:
# Fetch measured solar wind speed for each timestamp
# get_sw_speed uses speasy; falls back to default_vsw if unavailable
vsw_list = []
for i, t in enumerate(so_carr.obstime):
    v = get_sw_speed('Solar Orbiter', t.datetime, silent=False)
    vsw_list.append(v)
    if (i + 1) % 10 == 0:
        print(f"  Fetched {i + 1}/{len(so_carr.obstime)} Vsw values...")

vsw = np.array(vsw_list) * u.km / u.s
print(f"\nVsw range: {vsw.min():.0f} to {vsw.max():.0f}")
print(f"Vsw mean: {vsw.mean():.0f}")

  Fetched 10/10 Vsw values...

Vsw range: 338 km / s to 570 km / s
Vsw mean: 469 km / s


#### 2. Back-mapping from SC to Source Surface (SS)
- Base on SC locations and Vsw
- Assuming Parker spiral

In [ ]:
# Parker spiral backmapping: SC → Source Surface at 2.5 Rs
rss = 2.5

# backmapping_angle accepts arrays — vectorized, no loop needed
alpha = backmapping_angle(so_carr.radius, rss * u.R_sun, so_carr.lat, vsw, diff_rot=True).to(u.deg)

ss_lon = (Longitude(so_carr.lon) + alpha).wrap_at(360 * u.deg) # [0,360)
ss_lat = so_carr.lat  # latitude unchanged in radial Parker spiral

# Build summary DataFrame
df = pd.DataFrame({
    'time': [t.iso for t in so_carr.obstime],
    'sc_lon_deg': so_carr.lon.to(u.deg).value,
    'sc_lat_deg': so_carr.lat.to(u.deg).value,
    'sc_dist_AU': so_carr.radius.value,
    'vsw_km_s': vsw.value,
    'alpha_deg': alpha.value,
    'ss_lon_deg': ss_lon.to(u.deg).value,
    'ss_lat_deg': ss_lat.to(u.deg).value,
})
df

,time,sc_lon_deg,sc_lat_deg,sc_dist_AU,vsw_km_s,alpha_deg,ss_lon_deg,ss_lat_deg
0,2022-03-02 06:00:00.000,61.595567,-4.084653,0.561100,467.396912,29.410311,91.005878,-4.084653
1,2022-03-02 18:00:00.000,55.613333,-4.111851,0.554918,450.782104,30.149792,85.763125,-4.111851
2,2022-03-03 06:00:00.000,49.656244,-4.138073,0.548707,517.204285,25.976219,75.632462,-4.138073
3,2022-03-03 18:00:00.000,43.725276,-4.163221,0.542471,525.110962,25.286866,69.012142,-4.163221
4,2022-03-04 06:00:00.000,37.821449,-4.187192,0.536209,569.658203,23.033523,60.854972,-4.187192
5,2022-03-04 18:00:00.000,31.945827,-4.209871,0.529926,519.322876,24.962393,56.908220,-4.209871
6,2022-03-05 06:00:00.000,26.099521,-4.231135,0.523622,477.111145,26.839427,52.938949,-4.231135
7,2022-03-05 18:00:00.000,20.283690,-4.250851,0.517301,409.647675,30.872494,51.156184,-4.250851
8,2022-03-06 06:00:00.000,14.499541,-4.268874,0.510965,417.864441,29.885239,44.384780,-4.268874
9,2022-03-06 18:00:00.000,8.748330,-4.285050,0.504618,338.230957,36.451031,45.199361,-4.285050


### 3. PFSS

In [41]:
import os
from pathlib import Path
import sunpy.map
from sunkit_magex import pfss

DATA_DIR = (Path.cwd().parent / 'data').resolve()
RESULTS_DIR = (Path.cwd().parent / 'results').resolve()
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Data: {DATA_DIR}\nResults: {RESULTS_DIR}")

Data: /disk/plasma/xw2/PFSS/data
Results: /disk/plasma/xw2/PFSS/results


In [42]:
# Load HMI synoptic magnetogram (CR 2254, covers full date range)
hmi_synop = sunpy.map.Map(DATA_DIR / 'hmi.synoptic_mr_polfil_720s.2254.Mr_polfil.fits')

# Fix missing metadata
earth_coord = get_earth(hmi_synop.date)
hmi_synop.meta['dsun_obs'] = earth_coord.radius.to('m').value
hmi_synop.meta['hgln_obs'] = earth_coord.lon.to('deg').value
hmi_synop.meta['hglt_obs'] = earth_coord.lat.to('deg').value

# Resample and compute PFSS (single solve for all timestamps)
hmi_synop_resample = hmi_synop.resample([720, 360] * u.pix)

nrho = 25 # Number of radial grid points (higher = more accurate but slower)
pfss_input = pfss.Input(hmi_synop_resample, nrho, rss)
pfss_out = pfss.pfss(pfss_input)

print(f"PFSS coordinate_frame obstime: {pfss_out.coordinate_frame.obstime}")
print(f"PFSS grid shape: {pfss_out.grid.rg.shape}")

2026-04-03 18:17:38 - sunpy - INFO: Missing metadata for solar radius: assuming the standard radius of the photosphere.
2026-04-03 18:17:38 - sunpy - INFO: Missing metadata for solar radius: assuming the standard radius of the photosphere.


INFO: Missing metadata for solar radius: assuming the standard radius of the photosphere. [sunpy.map.mapbase]
INFO: Missing metadata for solar radius: assuming the standard radius of the photosphere. [sunpy.map.mapbase]
PFSS coordinate_frame obstime: 2022-02-21T20:07:23.000
PFSS grid shape: (26,)
